# 76장 · GitHub Actions의 작업 의존성과 실행 조건

PR에서 검사를 통과했는데 이미지가 발행되지 않았습니다. 어느 작업의 조건을 읽어야 할까요?

GitHub Actions에서 PR 검사와 main 병합 후 이미지 발행을 분리합니다. 이벤트 조건과 작업 간 의존성을 읽어 테스트가 성공해도 발행을 건너뛰는 경우를 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 PyYAML 6.0.3입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 워크플로 YAML의 needs와 if

step은 한 작업 안에서 차례로 진행합니다. job끼리 결과를 이어 쓰려면 needs로 의존 관계를 명시합니다. 아래는 실제 배포 양식에서 실행 조건을 뽑은 연습용 YAML입니다. 외부 저장소나 레지스트리를 변경하지 않습니다.

<!-- lesson-visual practice-space -->
**어떤 이벤트에서 publish가 실행될까요?**

워크플로의 needs와 if를 함께 적용합니다. 이벤트가 같아도 브랜치나 검사 결과가 다르면 어느 작업이 생략되는지 확인하세요.

네 조합 중 main에 푸시하고 선행 검사가 성공한 경우에만 publish가 실행됩니다. 실제 이미지는 발행하지 않는 조건 실험입니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter76/#practice-space)
<!-- /lesson-visual practice-space -->


### 이벤트와 게시 조건을 나눠 읽기

<p>on의 pull_request와 push는 워크플로를 시작하는 이벤트입니다. 그중 publish가 진행할 수 있는 경우는 push이고 ref가 refs/heads/main일 때입니다. PR에서도 검사는 실행 대상입니다.</p>


### YAML을 읽고 job 이름 찾기

<p>셀은 yaml.safe_load로 선언을 읽은 뒤 jobs 아래 publish의 needs 값을 꺼냅니다. 출력 check는 먼저 성공해야 하는 job의 이름입니다. 로컬에서 YAML을 읽는 동작과 GitHub가 job을 실행하는 동작은 별개의 단계입니다.</p>


### 셀 01 · needs의 선행 작업

check는 코드를 검사하고 publish는 검사한 코드의 이미지 후보를 담당합니다. 어느 작업이 다른 작업을 기다리는지 보세요.

아래 YAML은 needs로 검사와 게시 순서를 정하고 if로 게시 조건을 좁히는 발췌입니다. 여기에는 저장소 checkout이나 실제 이미지 빌드·발행 단계가 없습니다. 커밋 문자열 출력과 이미지 발행 성공은 구별해 기록하세요.


In [1]:
import yaml
workflow = yaml.safe_load("""
name: textbook-ci
"on": [pull_request, push]
jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - name: 단위 검사
        run: python -m unittest
  publish:
    needs: check
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - name: 후보 식별
        run: echo "$GITHUB_SHA"
""")
print(workflow["jobs"]["publish"]["needs"])

check


출력 check는 publish가 기다리는 job 이름입니다. YAML에서 두 job을 나란히 적는 순서는 실행 순서를 보장하지 않습니다. 선행 job이 성공해도 이벤트 조건은 다시 확인합니다.


### needs와 if가 함께 걸린 publish

<p>publish에는 선행 검사 check와 이벤트 조건이 모두 걸려 있습니다. 모의 함수 may_publish의 ok는 선행 검사 결과를 대신합니다. main에 push했더라도 ok가 False이면 세 조건을 모두 만족하지 못합니다.</p>

<h3>검사 통과 뒤 publish에서 실행할 한 줄</h3>

<p>pull_request에서는 check만 실행 대상입니다. main에 push한 경우에도 check가 실패하면 publish는 건너뜁니다. 두 조건이 맞을 때 이 YAML에서 실행할 줄은 echo "$GITHUB_SHA"입니다. 출력으로 얻는 것은 커밋 식별자입니다.</p>


## unittest로 복원 확인을 CI 검사로 옮기기

사람이 비교했던 본문과 출처를 다음 변경에서도 검사하도록 단위 테스트로 만듭니다. 실패 원인은 종료 코드와 테스트 이름에서 찾습니다.

노트북을 같은 수업 폴더에 모아 열거나 압축을 푼 장별 폴더에서 실행하세요. 시작 셀은 service-lab/restore.json을 먼저 엽니다. 파일이 없을 때만 셀에 적힌 복귀 예시를 읽습니다.

In [ ]:
from pathlib import Path
import json
here = Path.cwd()
# 각 장 폴더에서 열거나 노트북을 한 폴더에 모아 두어도 같은 실습 폴더를 씁니다.
course = here.parent if here.name in ("daily",) or (here.name.startswith("chapter") and here.name[7:].isdigit()) else here
lab = course / "service-lab"
lab.mkdir(exist_ok=True)
incoming = course / "service-lab/restore.json"
if incoming.exists():
    previous = json.loads(incoming.read_text(encoding="utf-8"))
    print("내 실습 파일", incoming.name)
else:
    previous = {'id': 'loan-demo',
     'expected': ['대출 기간은 14일입니다.', 'loan-01'],
     'restored': ['대출 기간은 14일입니다.', 'loan-01']}
    print("복귀 예시", incoming.name)

내 실습 파일 restore.json


### 복원 검사를 자동 테스트로 실행하기

In [ ]:
import subprocess, sys
ci_dir = lab / "ci"
ci_dir.mkdir(exist_ok=True)
(ci_dir / "restore.json").write_text(json.dumps(previous, ensure_ascii=False), encoding="utf-8")
test_code = """import json, unittest
from pathlib import Path
class RestoreTest(unittest.TestCase):
    def test_restored_row(self):
        record = json.loads(Path(__file__).with_name("restore.json").read_text(encoding="utf-8"))
        self.assertEqual(record["restored"], record["expected"])
"""
(ci_dir / "test_restore.py").write_text(test_code, encoding="utf-8")
run = subprocess.run([sys.executable, "-m", "unittest", "test_restore"], cwd=ci_dir, capture_output=True, text=True)
result = {"test": "test_restore.RestoreTest.test_restored_row", "exit_code": run.returncode, "source_id": previous["expected"][1]}
print(result["test"], run.returncode)

test_restore.RestoreTest.test_restored_row 0


로컬에서 실행한 unittest도 CI와 마찬가지로 성공하면 종료 코드 0을 반환합니다. 수업 저장소에서 service-lab/ci를 커밋하고 GitHub Actions의 실행 단계에 python -m unittest discover -s service-lab/ci를 지정해 같은 테스트를 찾으세요.

### 직접 해 보기

빈 폴더에서 찾은 테스트가 0건이어도 종료 코드가 0일 수 있습니다. 여기서 실제로 한 건을 실행했는지 stderr를 검사하세요.

In [ ]:
# run.stderr에서 Ran 1 test를 찾으세요.

<details class="notebook-answer"><summary>정답 코드와 실행 결과</summary><pre><code>print(&quot;Ran 1 test&quot; in run.stderr, run.returncode == 0)</code></pre><pre>True True</pre><p>실행 개수와 종료 코드를 함께 봐야 테스트를 찾지 못한 경우를 구분할 수 있습니다.</p></details>

### 77장에서 다시 열 파일

방금 계산한 값을 service-lab/ci.json에 저장합니다. 같은 장에서 조건을 바꾸고 다시 저장하면 다음 장도 그 값을 읽습니다.

In [ ]:
destination = lab / "ci.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination.name)

ci.json


### 과제 02 · if로 PR의 발행 제한

main에 push됐고 check가 성공하면 후보 단계가 실행되도록 판정 함수를 작성하세요. 같은 커밋이라도 PR 이벤트에서 후보 단계를 건너뛰는지 확인하세요.


In [2]:
def may_publish(event, ref, ok):
    # YAML의 if와 선행 검사 성공 조건을 합치세요.
    pass

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def may_publish(event, ref, ok):
    return event == &quot;push&quot; and ref == &quot;refs/heads/main&quot; and ok

print(may_publish(&quot;pull_request&quot;, &quot;refs/heads/main&quot;, True))
print(may_publish(&quot;push&quot;, &quot;refs/heads/main&quot;, True))
assert not may_publish(&quot;push&quot;, &quot;refs/heads/main&quot;, False)</code></pre>
<p>실행 결과</p><pre><code>False
True</code></pre>
<p>검사에 성공해도 PR이면 첫 값은 False입니다. main에 push한 같은 입력에서 check가 실패하면 마지막 단언이 그 경우도 검사합니다. 이 YAML의 후보 단계는 커밋 SHA를 출력합니다.</p>
</details>


#### 검사를 통과했는데 발행하지 않는 경우

<mark>선행 작업의 성공과 현재 이벤트가 발행 조건에 맞는지는 별개입니다.</mark> PR과 대상 브랜치의 이벤트를 각각 대입해 보세요. check가 끝났더라도 if 조건이 거짓이면 뒤의 발행 작업은 건너뜁니다.

선수 개념 · 71장 Dockerfile과 컨테이너 수명 주기

#### publish 작업이 실행되는 조건

```mermaid
flowchart TD
s["push 또는 pull_request"] --> c{"check 성공?"}
c -->|실패| n["publish 생략"]
c -->|성공| d{"main에 push?"}
d -->|아니오| n
d -->|예| p["publish 실행"]
```

check가 실패하면 종속 작업을 진행하지 않습니다. check가 성공한 경우에도 main 브랜치의 push인지 확인합니다.

## 검사 프로세스의 종료 코드

CI runner는 검사 프로그램의 종료 코드를 받습니다. 학생의 노트북에서 별도 Python 프로세스로 작은 검사를 돌리고, 실패한 결과에서 후보를 발행해도 될지 판단합니다.


### 포트 비교 실패가 게시 작업까지 전파되는 경로

같은 검사가 CI의 check 안에서 실패했다고 가정합니다. publish는 앞선 작업의 성공에 의존합니다.

1. 단위 검사 · 실제 비교 = 5432 != 55432
2. 종료 코드 · returncode = 1
3. check · 작업 결과 = failure
4. 게시 생략 · needs: check = skipped


### 셀 03 · 실패한 프로세스의 종료 코드

잘못된 기대값을 가진 단위 검사를 실행합니다. 자식 프로세스의 실패가 노트북 전체를 중단시키지 않도록 결과를 받습니다.


In [3]:
import subprocess, sys
test = """import unittest
class TestPort(unittest.TestCase):
    def test_container_port(self):
        self.assertEqual(5432, 55432)
unittest.main()
"""
run = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True)
print(run.returncode)
print("5432 != 55432" in run.stderr)

1
True


1과 True가 보입니다. 실행한 단위 검사가 호스트 포트와 컨테이너 포트를 혼동했습니다. 실제 workflow라면 실패한 검사와 연결된 후보 발행을 멈춰야 합니다. GitHub-hosted runner에서 실행한 기록은 아닙니다.


### 종료 코드와 실패 원인 읽기

<p>5432와 55432를 같다고 한 단위 검사는 실패합니다. returncode의 1은 실패가 발생했다는 신호이고, stderr에서 두 숫자의 불일치를 찾으면 어떤 비교가 실패했는지도 확인할 수 있습니다.</p>


### 과제 04 · 같은 단위 검사로 포트 수정 전후 비교

호스트에 공개한 55432를 내부 DB 포트로 반환하는 함수에 결함이 있습니다. 검사에서 기대하는 5432는 그대로 두고 함수만 수정하세요. 같은 단위 검사의 종료 코드가 어떻게 달라지는지 확인합니다.


In [ ]:
import subprocess, sys
test_case = """
import unittest
class PortTest(unittest.TestCase):
    def test_internal_port(self):
        self.assertEqual(internal_port(), 5432)
unittest.main()
"""
def run_check(implementation):
    result = subprocess.run([sys.executable, "-c", implementation + "\n" + test_case],
                            capture_output=True, text=True)
    return result.returncode

broken_port = "def internal_port(): return 55432"
# fixed_port 함수를 담은 문자열을 쓰고 같은 run_check로 변경 전후를 검사하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>import subprocess, sys
test_case = &quot;&quot;&quot;
import unittest
class PortTest(unittest.TestCase):
    def test_internal_port(self):
        self.assertEqual(internal_port(), 5432)
unittest.main()
&quot;&quot;&quot;
def run_check(implementation):
    result = subprocess.run([sys.executable, &quot;-c&quot;, implementation + &quot;\n&quot; + test_case],
                            capture_output=True, text=True)
    return result.returncode

broken_port = &quot;def internal_port(): return 55432&quot;

fixed_port = &quot;def internal_port(): return 5432&quot;
print(run_check(broken_port), run_check(fixed_port))</code></pre>
<p>실행 결과</p><pre><code>1 0</code></pre>
<p>수정한 것은 내부 포트를 반환하는 함수 한 줄입니다. 검사 코드는 두 실행에서 동일합니다. 기대값까지 55432로 바꿨다면 결함을 그대로 둔 채 검사가 통과했을 겁니다. 이 결과는 로컬 Python 프로세스의 종료 코드입니다.</p>
</details>


## CI가 실제로 회귀 검사를 발견하는지 확인하기

테스트 파일 경로를 잘못 넣으면 실행한 검사가 0개입니다. 임시 폴더에 작은 서비스 함수와 테스트 한 건을 준비해 GitHub Actions에서 사용할 것과 같은 unittest 명령을 실행합니다. 검사 미발견과 구현 오류의 종료 코드를 비교하세요.

### 셀 05 · 파일 이름이 패턴과 맞지 않을 때의 종료 코드

수업용 테스트 파일 이름은 test_port.py입니다. 처음에는 *_spec.py를 찾도록 설정하고 표준 오류에서 실행한 테스트 수를 읽습니다.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
import re
import subprocess
import sys

ci_area = TemporaryDirectory()
ci_root = Path(ci_area.name)
(ci_root / "service_port.py").write_text("def internal_port(): return 5432", encoding="utf-8")
(ci_root / "test_port.py").write_text(
    "import unittest\nfrom service_port import internal_port\n"
    "class PortTest(unittest.TestCase):\n"
    "    def test_db_port(self): self.assertEqual(internal_port(), 5432)\n", encoding="utf-8")

def run_suite(pattern):
    run = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", ".", "-p", pattern],
                         cwd=ci_root, capture_output=True, text=True)
    count = int(re.search(r"Ran (\d+) tests?", run.stderr).group(1))
    return run.returncode, count

print(run_suite("*_spec.py"))

(5, 0)


이 수업 커널에서는 (5, 0)이 나옵니다. 종료 코드 5는 unittest가 검사를 하나도 발견하지 못했다는 뜻입니다. 이때는 서비스 함수를 고치기 전에 파일 이름과 발견 패턴을 확인하세요.

### 과제 06 · 파일 하나를 발견하고 잘못된 구현에서 실패시키기

검사 패턴을 고쳐 현재 구현을 검사하세요. 다음으로 임시 service_port.py가 55432를 반환하도록 바꿔 같은 검사를 실행하세요. 두 실행의 종료 코드와 검사 수를 출력하세요.

In [5]:
# test_port.py를 발견할 패턴으로 run_suite를 실행하세요.
# 임시 파일에서 반환값을 55432로 바꾼 후 동일 명령을 다시 실행하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>pattern = &quot;test_*.py&quot;
print(run_suite(pattern))
(ci_root / &quot;service_port.py&quot;).write_text(&quot;def internal_port(): return 55432&quot;, encoding=&quot;utf-8&quot;)
print(run_suite(pattern))</code></pre>
<p>실행 결과</p><pre><code>(0, 1)
(1, 1)</code></pre>
<p>첫 결과 (0, 1)은 한 검사가 통과했다는 뜻입니다. 두 번째 (1, 1)은 같은 검사가 잘못된 값을 잡아낸 결과입니다. Actions의 check 작업에는 python -m unittest discover -s . -p "test_*.py"를 넣을 수 있습니다. publish가 check를 기다리게 하면 이 실패가 이미지 발행 이전에 드러납니다.</p>
</details>

<mark>종료 코드와 실행한 검사 수를 함께 확인하세요.</mark> 이 환경에서는 파일 이름이 맞지 않을 때 (5, 0), 테스트가 구현 오류를 잡았을 때 (1, 1)이 나왔습니다. 두 경우 모두 CI가 멈춰도 먼저 살펴볼 파일은 다릅니다.

검사에 사용한 파일과 명령, 이미지의 커밋 식별자를 함께 남기면 다음 장에서 어느 후보를 배포했는지 추적할 수 있습니다.

## 돌아보기

로컬에서 검사 프로세스의 종료 코드가 1이면 실패입니다. 예제 YAML대로라면 그 검사에 의존하는 publish가 진행할 수 없습니다. 실제 실행 기록에서는 check와 publish 두 job의 상태를 각각 읽습니다.

### 검사에서 게시 후보까지

**PR 또는 main push**　시작 사건과 브랜치를 확인합니다.

```
event/ref
```

**같은 코드 검사**　실패·취소·skip을 남깁니다.

```
tests conclusion
```

**게시 후보 여부**　PR은 여기서 게시 경로가 막힙니다.

```
main push + success
```

**같은 코드로 이미지 생성**　실제 실행 후 commit과 digest를 연결해야 합니다.

```
기록할 값: 검사한 커밋 / 이미지의 커밋
```


### 후보 식별 단계에서 남기는 Git SHA

<p>예제의 publish 단계는 GITHUB_SHA를 출력합니다. 이 값은 워크플로를 시작한 커밋을 식별합니다. 예제 YAML에는 코드 체크아웃 단계가 없으므로 실제 프로젝트의 테스트가 수집·실행됐는지는 이 출력만으로 확인할 수 없습니다. 여기서 이미지를 빌드하거나 레지스트리에 업로드하는 명령은 실행하지 않습니다.</p>
